# 06: Inference, Threats, and Robustness

A DiD estimate is not credible just because the regression runs.
Inference, pre-trends, placebo checks, robustness, and context
all matter.


In [1]:
from lite_setup import ensure_packages
await ensure_packages()

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import statsmodels.api as sm
import statsmodels.formula.api as smf

from checks import check_close, check_columns, check_same_estimate, check_sign
from did_utils import (
    did_2x2_table,
    manual_did,
    predicted_values_2x2,
    plot_group_trends,
    plot_did_counterfactual,
    make_event_dummies,
    extract_event_study_results,
    plot_event_study,
)


plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

DATA_DIR = Path("data")


Running outside JupyterLite; assuming packages are already installed.


In [2]:
df = pd.read_csv(DATA_DIR / "retail_rollout_panel.csv")
df.head()


,store_id,month,treated_store,post,did,sales,foot_traffic,ad_spend,region,store_size
0,ST001,1,1,0,0,144.41,890.4,57.43,North,small
1,ST001,2,1,0,0,147.98,889.8,66.85,North,small
2,ST001,3,1,0,0,146.85,952.9,60.71,North,small
3,ST001,4,1,0,0,156.35,954.0,62.04,North,small
4,ST001,5,1,0,0,149.40,951.4,73.73,North,small


In [3]:
formula = "sales ~ did + C(store_id) + C(month)"
model_nonrobust = smf.ols(formula, data=df).fit()
model_hc1 = smf.ols(formula, data=df).fit(cov_type="HC1")
model_cluster = smf.ols(formula, data=df).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["store_id"]},
)

comparison = pd.DataFrame(
    {
        "estimate": [
            model_nonrobust.params["did"],
            model_hc1.params["did"],
            model_cluster.params["did"],
        ],
        "std_error": [
            model_nonrobust.bse["did"],
            model_hc1.bse["did"],
            model_cluster.bse["did"],
        ],
    },
    index=["nonrobust", "HC1", "clustered by store"],
)
comparison


,estimate,std_error
nonrobust,8.074722,0.260223
HC1,8.074722,0.260223
clustered by store,8.074722,0.268274


Clustered standard errors are natural here because treatment is
assigned at the store level and repeated observations from the
same store are likely correlated. The large-sample justification
is about the number of clusters, not just the number of rows; in
this synthetic example there are 40 stores.


In [4]:
pre_only = df.query("month < 13").copy()
pre_only["placebo_post"] = (pre_only["month"] >= 10).astype(int)
pre_only["placebo_did"] = pre_only["treated_store"] * pre_only["placebo_post"]

placebo_model = smf.ols(
    "sales ~ placebo_did + C(store_id) + C(month)",
    data=pre_only,
).fit(cov_type="cluster", cov_kwds={"groups": pre_only["store_id"]})

placebo_model.params["placebo_did"], placebo_model.bse["placebo_did"]


(np.float64(-0.13555555555554585), np.float64(0.46257743897938364))

In [5]:
rows = []
for start, end in [(1, 24), (4, 21), (7, 18)]:
    part = df.query("@start <= month <= @end").copy()
    fit = smf.ols(
        "sales ~ did + foot_traffic + ad_spend + C(store_id) + C(month)",
        data=part,
    ).fit(cov_type="cluster", cov_kwds={"groups": part["store_id"]})
    rows.append({"window": f"{start}-{end}", "estimate": fit.params["did"], "std_error": fit.bse["did"]})

pd.DataFrame(rows)


,window,estimate,std_error
0,1-24,8.117461,0.189673
1,4-21,8.098140,0.217884
2,7-18,8.162223,0.264401


## Threats checklist

- Nonparallel pre-trends
- Anticipation before the official treatment date
- Spillovers from treated units to controls
- Simultaneous policies or shocks
- Composition changes
- Bad controls affected by treatment
- Treatment timing differences across units

Takeaway: a good DiD report states the estimate and also argues
why the identifying comparison is plausible.
